In [ ]:
# resume_fft_detector.py
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ds import (
    WatermarkOnTheFlyDataset,
    discover_dataset_files,
    make_train_image_augmentations,
    get_test_aug,
)
from model import make_model
from engine import train_epoch, safe_eval_call, train_epoch_psnr, safe_eval_call_psnr
import time
from model import load_checkpoint
from datetime import timedelta
from watermark import get_watermarking_mask, get_watermarking_pattern
import warnings

warnings.filterwarnings("ignore")

# ----------------- Config (adjust if needed) -----------------
DATA_DIR = "./watermark_dataset"
CHECKPOINT_PATH = "fft_detector_ckpt_epoch_100.pth"  # or "fft_detector_ckpt_full.pth"
CKPT_NAME = "PNSR_WM_PATTERN.pth"
# CHECKPOINT_PATH = None  # or "fft_detector_ckpt_full.pth"
BATCH_SIZE = 8
NUM_WORKERS = 0
TOTAL_NUM_EPOCHS = (
    200  # total epochs you want to reach (resume will continue until this)
)
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
SAVE_EVERY_EPOCHS = 1  # how often to save full checkpoint
IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = get_test_aug(IMAGE_SIZE)
INCLUDE_MASK_PATCH = False  # whether to include watermark mask and gt_patch in model input
INCLUDE_PSNR = True  # whether to include PSNR metric in dataset output

# Watermarking parameters (should match those used during watermark embedding)
W_MASK_SHAPE = "circle"
W_CHANNEL = 0
W_RADIUS = 8
W_STRENGTH = 0.9
W_PATTERN = "octoweb"
# -------------------------------------------------------------

# ---------------- reproducibility ----------------
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
# -------------------------------------------------

# ---------------- Load or define PIPE and TEXT_EMBEDDINGS ----------------
# Validate that PIPE and TEXT_EMBEDDINGS are present (or load them here)
try:
    import torch
    import diffusers
    from diffusers import DPMSolverMultistepScheduler
    from inverse_stable_diffusion import InversableStableDiffusionPipeline

    model_id = "stabilityai/stable-diffusion-2-1-base"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    scheduler = DPMSolverMultistepScheduler.from_pretrained(
        model_id, subfolder="scheduler"
    )
    pipe = InversableStableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch.float16,
        revision="fp16",
        verbose=False,
    )
    diffusers.utils.logging.disable_progress_bar()
    pipe.set_progress_bar_config(disable=True)
    pipe = pipe.to(device)

    TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
    PIPE = pipe  # make sure 'pipe' is in scope
except NameError:
    raise RuntimeError(
        "Please ensure `PIPE` and `TEXT_EMBEDDINGS` are available in the runtime before running resume script."
    )
# -----------------------------------------------------------------------


watermarking_mask = get_watermarking_mask(
    pipe.get_random_latents(),
    w_mask_shape=W_MASK_SHAPE,
    w_channel=W_CHANNEL,
    w_radius=W_RADIUS,
    device=device,
)

gt_patch = get_watermarking_pattern(
    pipe,
    w_seed=SEED,
    w_pattern=W_PATTERN,
    w_radius=W_RADIUS,
    device=device,
    strength=W_STRENGTH,
    shape=None,
)

#  ---------------- Prepare datasets and dataloaders ----------------
file_paths, labels = discover_dataset_files(DATA_DIR)
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]
train_ds = WatermarkOnTheFlyDataset(
    train_paths,
    train_labels,
    watermarking_mask=watermarking_mask,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=IMG_AUG,
    include_mask_patch=INCLUDE_MASK_PATCH,
    include_psnr=INCLUDE_PSNR,
    gt_patch=gt_patch,
    gt_psnr_return_prob=True,
)
val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=IMG_AUG,
    include_mask_patch=INCLUDE_MASK_PATCH,
    include_psnr=INCLUDE_PSNR,
    watermarking_mask=watermarking_mask,
    gt_patch=gt_patch,
    gt_psnr_return_prob=True,
)
val_ds_no_aug = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=None,
    include_mask_patch=INCLUDE_MASK_PATCH,
    include_psnr=INCLUDE_PSNR,
    watermarking_mask=watermarking_mask,
    gt_patch=gt_patch,
    gt_psnr_return_prob=True,
)
# -------------------------------------------------------------------

# infer input channels from a sample (may be expensive, but needed to create model)
sample_fft = train_ds[0][0]  # first in the tuple
in_ch = sample_fft.shape[0]

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
val_loader_no_aug = DataLoader(
    val_ds_no_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

# build model + optimizer + criterion
model = make_model(in_ch, include_psnr=INCLUDE_PSNR).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
crit = nn.CrossEntropyLoss()

model.base_model, start_epoch, best_val_loss, best_epoch = load_checkpoint(
    model.base_model, CHECKPOINT_PATH, DEVICE, opt=opt
)

Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  20%|██        | 1/5 [00:00<00:00,  9.09it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in director

Using PNSR wrapper for the model.
Loading checkpoint: fft_detector_ckpt_epoch_100.pth
Remaining keys in checkpoint:
odict_keys(['bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.bn1.num_batches_tracked', 'layer1.1.conv2.weight', 'layer1.1.bn2.weight', 'layer1.1.bn2.bias', 'layer1.1.bn2.running_mean', 'layer1.1.bn2.running_var', 'layer1.1.bn2.num_batches_tracked', 'layer2.0.conv1.weight', 'layer2.0.bn1.weight', 'layer2.0.bn1.bias', 'layer2.0.bn1.running_mean', 'layer2.0.bn1.running_var', 'layer

In [2]:
# training loop (resume) with pretty printing
total_start = time.time()
for epoch in range(start_epoch, TOTAL_NUM_EPOCHS + 1):
    epoch_start = time.time()
    print("=" * 80)
    print(
        f"Epoch {epoch:3d}/{TOTAL_NUM_EPOCHS:3d}    Time elapsed: {str(timedelta(seconds=int(time.time() - total_start)))}"
    )
    lr = opt.param_groups[0].get("lr", float("nan"))
    print(f"LR: {lr:.3e}")
    print("-" * 80)

    # train
    t0 = time.time()
    if INCLUDE_PSNR:
        train_loss = train_epoch_psnr(model, train_loader, opt, crit, device=DEVICE)
    else:
        train_loss = train_epoch(model, train_loader, opt, crit, device=DEVICE)
    t_train = time.time() - t0

    # validation (augmented) - informational
    if INCLUDE_PSNR:
        acc_aug, auc_aug, val_loss_aug = safe_eval_call_psnr(model, val_loader, crit, device)
        # validation (no aug) - primary criterion for saving
        acc_noaug, auc_noaug, val_loss_noaug = safe_eval_call_psnr(model, val_loader_no_aug, crit, device)
    else:
        acc_aug, auc_aug, val_loss_aug = safe_eval_call(model, val_loader, crit, device)
        # validation (no aug) - primary criterion for saving
        acc_noaug, auc_noaug, val_loss_noaug = safe_eval_call(model, val_loader_no_aug, crit, device)

    epoch_time = time.time() - epoch_start

    # pretty table-like summary
    # widths
    col1_w = 20
    col_w = 18
    print(
        f"{'Metric':<{col1_w}} {'Train':>{col_w}} {'Val (AUG)':>{col_w}} {'Val (NO-AUG)':>{col_w}}"
    )
    print("-" * (col1_w + col_w * 3 + 6))

    # Loss row
    print(
        f"{'Loss':<{col1_w}} "
        f"{train_loss:>{col_w}.4f} "
        f"{(val_loss_aug if not (val_loss_aug!=val_loss_aug) else float('nan')):>{col_w}.4f} "
        f"{(val_loss_noaug if not (val_loss_noaug!=val_loss_noaug) else float('nan')):>{col_w}.4f}"
    )

    # Accuracy row
    print(
        f"{'Accuracy':<{col1_w}} "
        f"{'--':>{col_w}} "
        f"{acc_aug:>{col_w}.4f} "
        f"{acc_noaug:>{col_w}.4f}"
    )

    # AUROC row
    print(
        f"{'AUROC':<{col1_w}} "
        f"{'--':>{col_w}} "
        f"{auc_aug:>{col_w}.4f} "
        f"{auc_noaug:>{col_w}.4f}"
    )

    print("-" * (col1_w + col_w * 3 + 6))
    print(
        f"Epoch time: {epoch_time:.1f}s (train: {t_train:.1f}s). Cumulative: {str(timedelta(seconds=int(time.time() - total_start)))}"
    )
    if best_epoch is not None:
        print(f"Best no-aug val loss so far: {best_val_loss:.6f} (epoch {best_epoch})")
    else:
        print(
            f"Best no-aug val loss so far: {best_val_loss if best_val_loss!=float('inf') else 'N/A'}"
        )

    # decide saving based on no-aug validation loss
    try:
        current_val_loss = float(val_loss_noaug)
    except Exception:
        current_val_loss = float("inf")

    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        best_epoch = epoch
        ckpt_dict = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
        }
        ckpt_name = CKPT_NAME
        torch.save(ckpt_dict, ckpt_name)
        print(
            f">>> Saved NEW BEST checkpoint: {ckpt_name} (best_val_loss={best_val_loss:.6f}, epoch={best_epoch})"
        )
    else:
        print(f"No improvement (current no-aug val loss {current_val_loss:.6f})")

print("=" * 80)
print("Training complete.")
print(f"Best no-aug val loss: {best_val_loss:.6f} (epoch {best_epoch})")

Epoch 101/200    Time elapsed: 0:00:00
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.6146             0.6321             0.4912
Accuracy                             --             0.6667             0.8000
AUROC                                --             0.7328             0.9000
--------------------------------------------------------------------------------
Epoch time: 1164.6s (train: 861.2s). Cumulative: 0:19:24
Best no-aug val loss so far: N/A
>>> Saved NEW BEST checkpoint: PNSR_AND_PATTERN.pth (best_val_loss=0.491166, epoch=101)
Epoch 102/200    Time elapsed: 0:19:24
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.5164             0.5475             0.4491
Accuracy                             --             0.7467             0.8000
AUROC                                --             0.8010             0.8937
--------------------------------------------------------------------------------
Epoch time: 1165.2s (train: 860.7s). Cumulative: 0:38:49
Best no-aug val loss so far: 0.491166 (epoch 101)
>>> Saved NEW BEST checkpoint: PNSR_AND_PATTERN.pth (best_val_loss=0.449139, epoch=102)
Epoch 103/200    Time elapsed: 0:38:49
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.4523             0.5236             0.4006
Accuracy                             --             0.7400             0.8267
AUROC                                --             0.8165             0.9080
--------------------------------------------------------------------------------
Epoch time: 1176.1s (train: 869.7s). Cumulative: 0:58:26
Best no-aug val loss so far: 0.449139 (epoch 102)
>>> Saved NEW BEST checkpoint: PNSR_AND_PATTERN.pth (best_val_loss=0.400555, epoch=103)
Epoch 104/200    Time elapsed: 0:58:26
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.3863             0.6123             0.4629
Accuracy                             --             0.6733             0.7600
AUROC                                --             0.7739             0.9032
--------------------------------------------------------------------------------
Epoch time: 1171.9s (train: 867.8s). Cumulative: 1:17:58
Best no-aug val loss so far: 0.400555 (epoch 103)
No improvement (current no-aug val loss 0.462944)
Epoch 105/200    Time elapsed: 1:17:58
LR: 2.000e-04
--------------------------------------------------------------------------------


...

Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.2174             0.4002             0.2932
Accuracy                             --             0.8333             0.8867
AUROC                                --             0.9189             0.9834
--------------------------------------------------------------------------------
Epoch time: 1479.2s (train: 1103.4s). Cumulative: 21:14:01
Best no-aug val loss so far: 0.203771 (epoch 132)
No improvement (current no-aug val loss 0.293236)
Epoch 157/200    Time elapsed: 21:14:01
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.2086             0.3368             0.2793
Accuracy                             --             0.8467             0.9000
AUROC                                --             0.9331             0.9848
--------------------------------------------------------------------------------
Epoch time: 1442.8s (train: 1064.2s). Cumulative: 21:38:03
Best no-aug val loss so far: 0.203771 (epoch 132)
No improvement (current no-aug val loss 0.279344)
Epoch 158/200    Time elapsed: 21:38:03
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.2312             0.4070             0.2838
Accuracy                             --             0.8133             0.8733
AUROC                                --             0.8989             0.9823
--------------------------------------------------------------------------------
Epoch time: 1458.6s (train: 1082.4s). Cumulative: 22:02:22
Best no-aug val loss so far: 0.203771 (epoch 132)
No improvement (current no-aug val loss 0.283835)
Epoch 159/200    Time elapsed: 22:02:22
LR: 2.000e-04
--------------------------------------------------------------------------------


Metric                            Train          Val (AUG)       Val (NO-AUG)
--------------------------------------------------------------------------------
Loss                             0.1907             0.4708             0.3245
Accuracy                             --             0.7667             0.8800
AUROC                                --             0.8795             0.9841
--------------------------------------------------------------------------------
Epoch time: 1448.9s (train: 1076.1s). Cumulative: 22:26:31
Best no-aug val loss so far: 0.203771 (epoch 132)
No improvement (current no-aug val loss 0.324493)
Epoch 160/200    Time elapsed: 22:26:31
LR: 2.000e-04
--------------------------------------------------------------------------------


KeyboardInterrupt: 

In [ ]:
# LR: 2.000e-04
# --------------------------------------------------------------------------------
                                                        
# Metric                            Train          Val (AUG)       Val (NO-AUG)
# --------------------------------------------------------------------------------
# Loss                             0.2427             0.3991             0.2483
# Accuracy                             --             0.8467             0.9267
# AUROC                                --             0.9162             0.9920
# --------------------------------------------------------------------------------
# Epoch time: 1289.2s (train: 925.8s). Cumulative: 10:22:51
# Best no-aug val loss so far: 0.225697 (epoch 116)
# No improvement (current no-aug val loss 0.248273)
# ================================================================================
# Epoch 132/200    Time elapsed: 10:22:51
# LR: 2.000e-04
# --------------------------------------------------------------------------------